# 05 - merged h5ad の確認

merge 結果の健全性チェック。obs の整合、`data_status`/`dataset_id` ごとの細胞数、遺伝子数、outer join による欠損の入り方を見る。**まだ解析しない。**

In [ ]:
import sys
from pathlib import Path

# config/dataset_manifest.yaml を持つ v2 ルートを探して src を import パスに追加
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

import manifest_utils as mf
man = mf.load_manifest()
paths = mf.project_paths(ROOT)
mf.ensure_dirs(paths)

import anndata as ad
import numpy as np
import scipy.sparse as sp

merged = {p.stem: ad.read_h5ad(p) for p in sorted(paths['merged'].glob('*.h5ad'))}
list(merged.keys())

In [ ]:
for name, a in merged.items():
    print(f'=== {name} ===  {a.n_obs} cells x {a.n_vars} genes')
    print('obs columns:', list(a.obs.columns))
    print()

## data_status / dataset_id ごとの細胞数

In [ ]:
for name, a in merged.items():
    print(f'=== {name} ===')
    if 'data_status' in a.obs:
        print(a.obs['data_status'].value_counts())
    print(a.obs['dataset_id'].value_counts())
    print()

## 遺伝子数・密度（outer join で 0/NaN が入る）

In [ ]:
for name, a in merged.items():
    X = a.X
    nnz = X.nnz if sp.issparse(X) else int(np.count_nonzero(X))
    print(f'{name}: {a.n_vars} genes, density={nnz / (a.n_obs * a.n_vars):.4f}')

## obs 列ごとの欠損

In [ ]:
for name, a in merged.items():
    print(f'=== {name} ===')
    miss = (a.obs.astype(str) == 'nan').sum() + a.obs.isna().sum()
    print(miss[miss > 0].sort_values(ascending=False).head(20))
    print()

ここまでが今回のスコープ。QC / 正規化 / クラスタリング / scVI は**まだ行わない**。